# Chapter 09 · 회귀 분석으로 주문 금액 예측하기

- 이름: 서지민 (의예과, 2024-14010)
- GitHub ID: jasminmuze
- 데이터: 수업 공통 프로젝트 쇼핑몰 데이터 (교육용 생성 데이터)
- 분석 단위: 주문 1건

이 과제에서는 주문 시점에 확인 가능한 달력 정보와 고객 특성만으로 주문 금액을 예측할 수 있는지 살펴봅니다. 높은 점수보다 데이터 관계 검증, 시간 순서 보존, 누수 방지, baseline 비교를 우선합니다.

## 분석 질문과 데이터 계획
1. 주문 월·요일, 고객 나이·성별·도시, 결제 수단으로 주문 금액을 예측할 수 있는가?
2. Linear Regression 또는 Random Forest가 평균 예측 Baseline보다 Train 기간의 TimeSeriesSplit MAE를 낮추는가?
3. Train CV에서 선택을 고정한 모델은 이후 기간의 Final Test에서 Baseline보다 나은가?

| 데이터 | 원본 행 수(헤더 제외) | 주요 변수 |
|---|---:|---|
| 고객 | 150 | customer_id, gender, age, city |
| 상품 | 100 | product_id, category, price |
| 주문 | 300 | order_id, customer_id, order_date, payment_method, order_status |
| 주문 상세 | 764 | order_item_id, order_id, product_id, quantity, unit_price |

데이터 출처는 수업 공개 저장소의 [공통 Raw 데이터](https://github.com/GilbertMoon/llm-data-analysis-course/tree/main/data/raw)이며, 이번 장에서는 Chapter05 오류 탐지용 전용 Raw를 사용하지 않습니다. 주문 금액은 quantity 곱하기 unit_price를 주문별로 합산해 Target으로 만듭니다. 이 데이터는 교육용 생성 자료이므로 실제 고객이나 시장의 구매 행동을 대표하지 않습니다.

## 예측 시점 및 누수 방지 기준
예측 시점은 주문 상세가 만들어지기 전으로 가정합니다. 그래서 order_month, order_dayofweek, age, payment_method, gender, city만 사용합니다. Target 재료(quantity, unit_price, line_total), Target 파생·대리변수, 사후정보(order_status), 식별자(order_id, customer_id, product_id)는 모델 입력에서 제외합니다.

주문과 주문별 Target은 일대일, 주문과 고객은 다대일 관계를 검증합니다. 관계가 깨지면 조용히 행을 버리지 않고 분석을 중단합니다.

## 데이터 준비와 재현 방법
저장소 루트에서 먼저 아래 명령을 실행합니다. 이 단계는 공통 Raw를 전처리하고 외래키 관계가 유효한지 검사합니다.

    pip install -r chapter09/requirements.txt
    python scripts/prepare_ch09_data.py

노트북 실행은 저장소 루트에서 Jupyter를 열고 chapter09/chapter09.ipynb를 실행합니다. 전체 과정을 스크립트로 재현하려면 다음을 실행합니다.

    python scripts/run_regression_analysis.py

## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [ ]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

## 2. 검증된 전처리 데이터를 불러옵니다

In [ ]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [ ]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

## 4. Feature Audit을 확인합니다

In [ ]:
feature_audit = build_feature_audit()
display(feature_audit)

## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [ ]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

In [ ]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [ ]:
models = make_regression_models(random_state=42)
print(list(models))

## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [ ]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [ ]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [ ]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 10. 실제값·예측값·잔차를 진단합니다

In [ ]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

## 11. 자동 Validation Evidence를 만듭니다

In [ ]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [ ]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [ ]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

## 14. 해석

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

## 결과 해석 원칙
- Train 기간의 TimeSeriesSplit 평균 MAE만으로 비베이스라인 후보를 선택하고, 선택을 고정한 뒤 Final Test를 확인합니다.
- Final Test에서는 고정 모델과 Baseline Mean만 비교합니다. Test를 본 뒤 후보를 바꾸지 않습니다.
- MAE·RMSE·R²와 Baseline 대비 MAE 개선율은 실행된 표에서 확인합니다. R²가 음수이거나 고정 모델이 Baseline보다 나쁘면 그 결과를 그대로 기록하고, 현재 정보만으로는 모델 사용을 보류합니다.
- 오차가 큰 주문은 내부 진단에서만 ID로 확인하고, 공개 결과에는 식별자를 넣지 않습니다. 관찰된 오차와 가능한 원인, 추가로 필요한 데이터를 구분합니다.
- 소규모 교육용 생성 데이터의 결과를 실제 소비자나 실제 매출로 일반화하거나 인과관계로 해석하지 않습니다.

## 생성형 AI 활용 및 검증
생성형 AI는 과제 지침을 체크리스트로 정리하고 회귀 분석 코드 구조와 설명 초안을 만드는 데 사용했습니다. 모델 성능과 결론은 AI의 제안으로 확정하지 않습니다. 실제 데이터에서 FK 관계, 허용·금지 feature, 날짜 분할, Train CV의 선택 결과, Final Test 검증 표가 모두 확인되는지 코드 실행 결과로 대조합니다. 검증이 실패하면 결과 해석을 보류합니다.

## 15. 완료 체크

- [ ] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [ ] Target과 예측 시점을 설명할 수 있다.
- [ ] 금지 feature가 입력에 없음을 확인했다.
- [ ] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [ ] 전처리는 Pipeline 내부에서 학습된다.
- [ ] Train TimeSeriesSplit으로 후보를 비교했다.
- [ ] Final Test 전에 Selected Model을 고정했다.
- [ ] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [ ] MAE, RMSE, R²를 함께 읽었다.
- [ ] 내부 식별자 결과와 공개 보고서를 구분했다.
- [ ] Validation Evidence가 모두 PASS다.